# 01 Data quality

Part 1 (1.1-1.8) counts each problem in rows and THB. Part 2 (2.1-2.4) cleans with `src/load_clean.py` and saves to `data/processed/`.

`data/raw/` is never changed.

## 1. What is wrong with the raw data?

### 1.1 What is in the workbook?

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# Works from the project root or from notebooks/.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_FILE = ROOT / "data" / "raw" / "FruitBlend24_Intern_Case_Data.xlsx"
assert RAW_FILE.exists(), f"Raw file not found: {RAW_FILE}"

# Load all sheets, so a sheet we did not expect still shows up.
sheets = pd.read_excel(RAW_FILE, sheet_name=None)

print(f"pandas {pd.__version__} | file: {RAW_FILE.name}")
print(f"Number of sheets: {len(sheets)}\n")
shape_table = pd.DataFrame(
    {"rows": [df.shape[0] for df in sheets.values()],
     "columns": [df.shape[1] for df in sheets.values()]},
    index=pd.Index(list(sheets.keys()), name="sheet"),
)
print(shape_table.to_string())

pandas 3.0.6 | file: FruitBlend24_Intern_Case_Data.xlsx
Number of sheets: 11

                            rows  columns
sheet                                    
README                        28        1
orders_hourly             124397        9
weekly_fruit_cost            239        3
platform_commission_rate       2        3
waste_daily                 6570        5
cost_assumptions              12        4
rate_code_dim                  4        5
platform_dim                   2        2
sku_code_dim                   5        2
monthly_budget                12        4
data_dictionary               33        3


**11 sheets: the 9 data sheets + README + data_dictionary.** `orders_hourly` is the main table (124,397 rows).

`waste_daily` has 6,570 rows, expected 6,576 (4 SKUs x 4 kitchens x 365 days + berry 4 x 184) -> 6 missing, see 1.2.

`weekly_fruit_cost` 239 rows = 53 weeks x 4 SKUs + 27 berry weeks. OK.

### 1.2 Which columns have missing values, odd types or duplicates?

In [2]:
def profile(df: pd.DataFrame, name: str) -> pd.DataFrame:
    """One row per column: type, missing, distinct, example, min, max."""
    rows = []
    for col in df.columns:
        s = df[col]
        non_null = s.dropna()
        # min/max fail on columns that mix numbers and text. That is a finding, so record it.
        try:
            col_min, col_max = non_null.min(), non_null.max()
        except TypeError:
            col_min, col_max = "mixed types", "mixed types"
        rows.append({
            "column": col,
            "dtype": str(s.dtype),
            "n_missing": int(s.isna().sum()),
            "pct_missing": round(s.isna().mean() * 100, 2),
            "n_unique": int(s.nunique(dropna=True)),
            "example": non_null.iloc[0] if len(non_null) else None,
            "min": col_min,
            "max": col_max,
        })
    # duplicated() counts only the extra copies, not the first row.
    n_dup = int(df.duplicated().sum())
    print(f"[{name}] {len(df):,} rows | exact duplicate rows (extra copies): {n_dup:,}")
    return pd.DataFrame(rows).set_index("column")


profiles = {}
for sheet_name, df in sheets.items():
    profiles[sheet_name] = profile(df, sheet_name)
    display(profiles[sheet_name])

[README] 28 rows | exact duplicate rows (extra copies): 4


,dtype,n_missing,pct_missing,n_unique,example,min,max
column,,,,,,,
FruitBlend24 — Intern Data Case,str,5,17.86,23,Business context: FruitBlend24 sells fresh fru...,NOTE: item_name_...,yourself from orders_hourly + weekly_fruit_cos...


[orders_hourly] 124,397 rows | exact duplicate rows (extra copies): 372


,dtype,n_missing,pct_missing,n_unique,example,min,max
column,,,,,,,
date,datetime64[us],0,0.0,365,2026-05-23 00:00:00,2025-09-01 00:00:00,2026-08-31 00:00:00
hour,int64,0,0.0,24,18,0,23
day_of_week,int64,0,0.0,7,5,0,6
kitchen,str,0,0.0,4,Pattaya_Central,BKK_Ladprao,Pattaya_Central
item_name_raw,str,0,0.0,31,เสาวรสปั่น,Combo Set A (2 แก้ว),แตงโมปั่นสด
price_thb,float64,0,0.0,20,65.0,0.0,99.0
units_sold,int64,0,0.0,53,8,1,67
gross_revenue_thb,float64,0,0.0,368,520.0,0.0,3015.0
rate_code,str,0,0.0,16,LM-RC000,GB-RC000,LM-RC103


[weekly_fruit_cost] 239 rows | exact duplicate rows (extra copies): 0


...

,dtype,n_missing,pct_missing,n_unique,example,min,max
column,,,,,,,
sku_code,str,0,0.0,5,WM,GV,WM
sku,str,0,0.0,5,Watermelon,Guava,Watermelon


[monthly_budget] 12 rows | exact duplicate rows (extra copies): 0


,dtype,n_missing,pct_missing,n_unique,example,min,max
column,,,,,,,
month,str,0,0.0,12,2025-09-01,2025-09-01,2026-08-01
budget_revenue_thb,int64,0,0.0,1,2700000,2700000,2700000
budget_gross_profit_thb,int64,0,0.0,1,700000,700000,700000
note,str,0,0.0,1,Flat monthly run-rate target set by management...,Flat monthly run-rate target set by management...,Flat monthly run-rate target set by management...


[data_dictionary] 33 rows | exact duplicate rows (extra copies): 0


,dtype,n_missing,pct_missing,n_unique,example,min,max
column,,,,,,,
sheet,str,0,0.0,8,orders_hourly,monthly_budget,weekly_fruit_cost
column,str,0,0.0,27,date,base_code,week_start
description,str,0,0.0,31,Calendar date,0=Monday ... 6=Sunday,units_sold x price_thb


**`waste_daily`: 4 missing waste costs, all MixedBerryPremium on 2026-03-01 (launch day, Sunday).** Berry fruit cost starts week 2026-03-02, so launch-day orders will miss fruit cost too -> see below. 6 kitchen-days missing -> see below.

**`cost_assumptions` is 2 tables in 1 sheet:** rows 0-4 cost per cup, rows 7-11 overhead per kitchen (row 7 = 2nd header) -> read separately. Dates in `weekly_fruit_cost`, `waste_daily`, `monthly_budget` are text -> parse before joins.

`orders_hourly`: 372 duplicate copies, 31 item names, min price 0 -> 1.4 and 1.8. No other missing values. Units 1-67, fruit cost 6.9-54 THB/cup. OK.

#### 1.2.1 Which `waste_daily` kitchen-days are missing?

In [3]:
waste = sheets["waste_daily"].copy()
waste["date"] = pd.to_datetime(waste["date"])  # stored as text

# Expected = every kitchen x SKU x day. Berry only from launch day, so earlier days are not "missing".
days = pd.date_range("2025-09-01", "2026-08-31", freq="D")
expected = pd.MultiIndex.from_product(
    [days, waste["kitchen"].unique(), waste["sku"].unique()],
    names=["date", "kitchen", "sku"],
).to_frame(index=False)
expected = expected[~((expected["sku"] == "MixedBerryPremium") & (expected["date"] < "2026-03-01"))]

check = expected.merge(waste, on=["date", "kitchen", "sku"], how="left",
                       indicator=True, validate="one_to_one")
missing = check.loc[check["_merge"] == "left_only", ["date", "kitchen", "sku"]]
print(f"Expected {len(expected):,} | actual {len(waste):,} | missing {len(missing)}")
display(missing)

# Size the gap with the average waste of the same kitchen and SKU.
avg = waste.groupby(["kitchen", "sku"])[["units_wasted", "est_waste_cost_thb"]].mean()
est = missing.join(avg, on=["kitchen", "sku"])
gap_thb = est["est_waste_cost_thb"].sum()
print(f"Estimated gap: {est['units_wasted'].sum():.0f} cups, {gap_thb:,.0f} THB "
      f"({gap_thb / waste['est_waste_cost_thb'].sum():.2%} of waste cost)")

assert len(expected) - len(missing) == len(waste), "waste rows outside the expected grid"

Expected 6,576 | actual 6,570 | missing 6


,date,kitchen,sku
180,2025-09-12,BKK_Sukhumvit,Guava
681,2025-10-13,Pattaya_Beach,PassionFruit
1380,2025-11-26,BKK_Sukhumvit,Guava
3206,2026-03-16,Pattaya_Beach,Guava
4479,2026-05-19,BKK_Ladprao,Watermelon
5779,2026-07-23,BKK_Ladprao,Watermelon


Estimated gap: 27 cups, 457 THB (0.05% of waste cost)


**6 kitchen-days missing, about 27 cups / 457 THB (0.05% of waste cost).** Guava 3, Watermelon 2, PassionFruit 1, all on different dates -> looks like random missed entries, not an outage.

Decision: fill with the kitchen x SKU mean in cleaning. P&L waste = raw + 457 THB filled (see `docs/decisions.md`).

#### 1.2.2 What does the berry launch day miss?

In [4]:
fruit_cost = sheets["weekly_fruit_cost"].copy()
fruit_cost["week_start"] = pd.to_datetime(fruit_cost["week_start"])  # stored as text
first_week = fruit_cost.loc[fruit_cost["sku"] == "MixedBerryPremium", "week_start"].min()

berry = waste[waste["sku"] == "MixedBerryPremium"]
launch = berry[berry["est_waste_cost_thb"].isna()]
print(f"Berry launch: {launch['date'].min().date()} | first berry cost week: {first_week.date()}")
display(launch[["date", "kitchen", "units_wasted"]])

# Weighted cost per cup in the first cost week (sum cost / sum cups, not mean of rows).
week1 = berry[berry["date"].between(first_week, first_week + pd.Timedelta(days=6))]
cost_per_cup = week1["est_waste_cost_thb"].sum() / week1["units_wasted"].sum()
launch_gap = launch["units_wasted"].sum() * cost_per_cup
print(f"Week {first_week.date()}: {cost_per_cup:.2f} THB per wasted cup")
print(f"Launch-day gap: {launch['units_wasted'].sum()} cups x {cost_per_cup:.2f} = {launch_gap:,.0f} THB")

assert (launch["date"] < first_week).all(), "blank cost is not before the first cost week"

# Berry orders on launch day miss fruit cost too (raw rows, before dedup).
o_raw = sheets["orders_hourly"]
launch_orders = o_raw[o_raw["item_name_raw"].str.contains("berry|เบอร์รี่", case=False)
                      & (o_raw["date"] == "2026-03-01")]
launch_dedup = launch_orders.drop_duplicates()
print(f"Berry orders on launch day: {len(launch_orders)} rows, {launch_orders['units_sold'].sum()} cups, "
      f"{launch_orders['gross_revenue_thb'].sum():,.0f} THB "
      f"| after dropping copies: {launch_dedup['units_sold'].sum()} cups, "
      f"{launch_dedup['gross_revenue_thb'].sum():,.0f} THB")

Berry launch: 2026-03-01 | first berry cost week: 2026-03-02


,date,kitchen,units_wasted
2894,2026-03-01,BKK_Ladprao,27
2899,2026-03-01,BKK_Sukhumvit,5
2904,2026-03-01,Pattaya_Beach,28
2909,2026-03-01,Pattaya_Central,36


Week 2026-03-02: 41.69 THB per wasted cup
Launch-day gap: 96 cups x 41.69 = 4,002 THB
Berry orders on launch day: 75 rows, 322 cups, 28,658 THB | after dropping copies: 322 cups, 28,658 THB


**Berry launch day (2026-03-01): 96 wasted cups with no cost (about 4.0k THB), and 322 cups sold (28.7k THB) with no fruit cost.** `.sum()` skips blanks, so without a fix waste is too low and launch-day CM is too high.

Decision: use the 2026-03-02 week cost for launch day, for both waste and orders (see `docs/decisions.md`).

### 1.3 Does `orders_hourly` cover every day, kitchen and hour?

In [5]:
orders = sheets["orders_hourly"]

d_min, d_max = orders["date"].min(), orders["date"].max()
n_days = orders["date"].nunique()
# Compare with a full calendar to catch missing days.
all_days = pd.date_range(d_min, d_max, freq="D")
missing_days = all_days.difference(orders["date"].unique())
print(f"Date range: {d_min.date()} to {d_max.date()}")
print(f"Days with data: {n_days} (expected 365) | days missing from calendar: {len(missing_days)}")

print("\nRows per kitchen:")
print(orders["kitchen"].value_counts().to_string())

print(f"\nHour min/max: {orders['hour'].min()} / {orders['hour'].max()} "
      f"| distinct hours: {orders['hour'].nunique()}")

Date range: 2025-09-01 to 2026-08-31
Days with data: 365 (expected 365) | days missing from calendar: 0

Rows per kitchen:
kitchen
Pattaya_Central    33202
Pattaya_Beach      32200
BKK_Sukhumvit      31357
BKK_Ladprao        27638

Hour min/max: 0 / 23 | distinct hours: 24


Dates 2025-09-01 to 2026-08-31, 365 days, no gaps. 4 kitchens (27.6k-33.2k rows each), hours 0-23. OK.

### 1.4 How many names are there for 5 menu items?

In [6]:
# Sort by revenue, not rows: a rare name can still carry a lot of money.
items = (
    orders.groupby("item_name_raw", dropna=False)
    .agg(rows=("item_name_raw", "size"),
         units=("units_sold", "sum"),
         revenue_thb=("gross_revenue_thb", "sum"))
    .sort_values("revenue_thb", ascending=False)
)
items["pct_revenue"] = (items["revenue_thb"] / items["revenue_thb"].sum() * 100).round(3)
print(f"Distinct item_name_raw values: {len(items)}")
with pd.option_context("display.max_rows", None, "display.float_format", "{:,.2f}".format):
    display(items)

assert np.isclose(items["revenue_thb"].sum(), orders["gross_revenue_thb"].sum())

Distinct item_name_raw values: 31


,rows,units,revenue_thb,pct_revenue
item_name_raw,,,,
Watermelon Ice,4560,36416,"1,607,353.65",5.14
watermelon ice,4436,35379,"1,560,957.30",4.99
Watermelon smoothy,4543,35228,"1,556,443.35",4.98
Watermelon Smoothie,4530,34910,"1,540,492.20",4.92
แตงโมปั่น (ไซส์ M),4379,34804,"1,535,660.10",4.91
แตงโมปั่น,4381,34722,"1,531,074.60",4.89
แตงโมปั่นสด,4383,34233,"1,511,953.20",4.83
สับปะรดปั่น,4994,27643,"1,330,728.77",4.25
Pineapple Ice,4987,27412,"1,322,693.75",4.23


In [7]:
# Orange test menu, empty cup, 2-cup combo. Names copied from the table above.
NON_MENU = ["น้ำส้มปั่น (เมนูทดลอง)", "แก้วเปล่า", "Combo Set A (2 แก้ว)"]
non_menu = orders[orders["item_name_raw"].isin(NON_MENU)]
display(non_menu.groupby("item_name_raw").agg(
    rows=("units_sold", "size"),
    units=("units_sold", "sum"),
    revenue_thb=("gross_revenue_thb", "sum"),
    prices=("price_thb", lambda s: sorted(s.unique())),
))
non_menu_rev = non_menu["gross_revenue_thb"].sum()
print(f"Non-menu total: {len(non_menu)} rows, {non_menu_rev:,.0f} THB "
      f"({non_menu_rev / orders['gross_revenue_thb'].sum():.2%} of revenue)")

is_berry = orders["item_name_raw"].str.contains("berry|เบอร์รี่", case=False)
print("\nFirst sale date of each berry name (launch = 2026-03-01):")
print(orders[is_berry].groupby("item_name_raw")["date"].min().dt.date.to_string())

print(f"\nNon-menu rate codes: {sorted(non_menu['rate_code'].unique())} "
      f"| cups per row: {non_menu['units_sold'].min()}-{non_menu['units_sold'].max()} "
      f"(menu rows: up to {orders.loc[~orders['item_name_raw'].isin(NON_MENU), 'units_sold'].max()})")
assert (orders[is_berry].groupby("item_name_raw")["date"].min() == "2026-03-01").all()

,rows,units,revenue_thb,prices
item_name_raw,,,,
Combo Set A (2 แก้ว),122,237,11879.0,"[0.0, 35.0, 59.0, 99.0]"
น้ำส้มปั่น (เมนูทดลอง),136,281,14870.0,"[0.0, 35.0, 59.0, 99.0]"
แก้วเปล่า,116,250,12289.0,"[0.0, 35.0, 59.0, 99.0]"


Non-menu total: 374 rows, 39,038 THB (0.12% of revenue)

First sale date of each berry name (launch = 2026-03-01):
item_name_raw
Mixed Berry (Premium)     2026-03-01
Mixed Berry Premium       2026-03-01
berry premium smoothie    2026-03-01
เบอร์รี่ปั่นพรีเมียม      2026-03-01
เบอร์รี่รวมพรีเมียม       2026-03-01

Non-menu rate codes: ['GB-RC000', 'LM-RC000'] | cups per row: 1-3 (menu rows: up to 67)


**31 raw names: 28 look like the 5 menu items** (by eye: Watermelon 7, Pineapple 6, the other 3 have 5 each; English, Thai, typos, case, 1 trailing space). Confirm in `item_mapping.csv`.

**3 are not menu items: orange test menu, empty cup, Combo Set A = 374 rows, 768 cups, 39k THB (0.12%).** All 3 have the same prices 0/35/59/99 at full-price code RC000, and only 1-3 cups per row -> test or junk entries with no cost data. Decision: DROP in mapping.

All 5 berry names start 2026-03-01 = launch date. OK.

### 1.5 Which rate codes are used?

In [8]:
rate_counts = orders["rate_code"].value_counts(dropna=False).rename("rows").to_frame()
rate_counts["pct_rows"] = (rate_counts["rows"] / len(orders) * 100).round(2)
print(f"Distinct rate_code values: {len(rate_counts)}")
with pd.option_context("display.max_rows", None):
    display(rate_counts)

Distinct rate_code values: 16


,rows,pct_rows
rate_code,,
LM-RC000,62961,50.61
GB-RC000,50765,40.81
LM-RC103,2426,1.95
LM-RC102,2309,1.86
GB-RC103,1810,1.46
GB-RC102,1765,1.42
LM-RC101-WM,476,0.38
LM-RC101-PF,324,0.26
LM-RC101-GV,299,0.24


**16 rate codes, all match `PLATFORM-BASECODE[-SKU]`.** RC000 is 91.4% of rows (LM 50.6%, GB 40.8%).

RC101 always has a SKU part (WM, PA, GV, PF, MB). OK.

RC101 SKU part vs the row's menu, after mapping: 0 mismatches (see 2.1).

### 1.6 What do the code tables say?

In [9]:
# Show full text: the conditions column changes how we read each code.
with pd.option_context("display.max_colwidth", None, "display.max_rows", None):
    for sheet_name in ["rate_code_dim", "platform_dim", "sku_code_dim", "platform_commission_rate"]:
        print(f"\n{sheet_name} {sheets[sheet_name].shape}")
        display(sheets[sheet_name])


rate_code_dim (4, 5)


,base_code,rate_code_name,discount_pct,code_pattern,conditions
0,RC000,Standard,0.00,{PLATFORM}-RC000,No promotion. Regular menu price.
1,RC101,Bundle Deal,0.15,{PLATFORM}-RC101-{SKU},"Randomly scheduled single-SKU bundle discount, ~1 in 10 days per SKU."
2,RC102,Double Day New User,0.20,{PLATFORM}-RC102,"New-user acquisition campaign run company-wide on 'double dates' (1/1, 2/2, ... 12/12)."
3,RC103,Seasonal Push,0.12,{PLATFORM}-RC103,"Company-wide rainy-season (Jun-Oct) demand-boost discount, run on scattered random days."



platform_dim (2, 2)


,platform_code,platform_name
0,LM,LINE MAN
1,GB,Grab



sku_code_dim (5, 2)


,sku_code,sku
0,WM,Watermelon
1,PA,Pineapple
2,GV,Guava
3,PF,PassionFruit
4,MB,MixedBerryPremium



platform_commission_rate (2, 3)


,platform_code,platform_name,commission_rate
0,LM,LINE MAN,0.30
1,GB,Grab,0.24


In [10]:
base_code = orders["rate_code"].str.split("-").str[1]

rc103_dates = orders.loc[base_code == "RC103", "date"]
print("RC103 rows by month (should be Jun-Oct only):")
print(rc103_dates.dt.to_period("M").value_counts().sort_index().to_string())

rc102_dates = orders.loc[base_code == "RC102", "date"]
off_double = rc102_dates[rc102_dates.dt.month != rc102_dates.dt.day]
print(f"\nRC102: {rc102_dates.dt.date.nunique()} dates | rows not on a double date: {len(off_double)}")

RC103 rows by month (should be Jun-Oct only):
date
2025-09     892
2025-10     612
2026-06     985
2026-07     690
2026-08    1057
Freq: M



RC102: 12 dates | rows not on a double date: 0


**RC000 = 0% discount, "regular menu price" -> likely the normal price.** Promos: RC101 bundle 15% (1 SKU, ~1 in 10 days), RC102 new user 20% (1/1, 2/2 ... 12/12), RC103 seasonal 12% (Jun-Oct).

**RC102 only on its 12 double dates, RC103 only in Sep-Oct 2025 and Jun-Aug 2026.** Both match the conditions. Commission: LINE MAN 30%, Grab 24%. Platform and SKU codes match `rate_code`. OK.

? RC102 is for new users, but there is no user column -> we can measure extra sales, not new users.

### 1.7 What do the cost and budget tables say?

In [11]:
# Show full text: the budget note decides what "gross profit" means.
with pd.option_context("display.max_colwidth", None, "display.max_rows", None):
    for sheet_name in ["cost_assumptions", "monthly_budget"]:
        print(f"\n{sheet_name} {sheets[sheet_name].shape}")
        display(sheets[sheet_name])


cost_assumptions (12, 4)


,sku,packaging_cost_per_cup_thb,labor_cost_per_cup_thb,base_price_thb
0,Watermelon,6,5,45.0
1,Pineapple,6,5,49.0
2,Guava,6,5.5,49.0
3,PassionFruit,6,6,65.0
4,MixedBerryPremium,8,7,89.0
5,NaN,NaN,NaN,NaN
6,"Fixed monthly overhead per central kitchen (rent, utilities, base staffing):",NaN,NaN,NaN
7,kitchen,fixed_monthly_overhead_thb,note,NaN
8,Pattaya_Central,180000,"rent + utilities + base staff, central kitchen",NaN
9,Pattaya_Beach,150000,"rent + utilities + base staff, central kitchen",NaN



monthly_budget (12, 4)


,month,budget_revenue_thb,budget_gross_profit_thb,note
0,2025-09-01,2700000,700000,Flat monthly run-rate target set by management at the start of the fiscal year; does not adjust for seasonality or input-cost volatility.
1,2025-10-01,2700000,700000,Flat monthly run-rate target set by management at the start of the fiscal year; does not adjust for seasonality or input-cost volatility.
2,2025-11-01,2700000,700000,Flat monthly run-rate target set by management at the start of the fiscal year; does not adjust for seasonality or input-cost volatility.
3,2025-12-01,2700000,700000,Flat monthly run-rate target set by management at the start of the fiscal year; does not adjust for seasonality or input-cost volatility.
4,2026-01-01,2700000,700000,Flat monthly run-rate target set by management at the start of the fiscal year; does not adjust for seasonality or input-cost volatility.
5,2026-02-01,2700000,700000,Flat monthly run-rate target set by management at the start of the fiscal year; does not adjust for seasonality or input-cost volatility.
6,2026-03-01,2700000,700000,Flat monthly run-rate target set by management at the start of the fiscal year; does not adjust for seasonality or input-cost volatility.
7,2026-04-01,2700000,700000,Flat monthly run-rate target set by management at the start of the fiscal year; does not adjust for seasonality or input-cost volatility.
8,2026-05-01,2700000,700000,Flat monthly run-rate target set by management at the start of the fiscal year; does not adjust for seasonality or input-cost volatility.
9,2026-06-01,2700000,700000,Flat monthly run-rate target set by management at the start of the fiscal year; does not adjust for seasonality or input-cost volatility.


**Budget: flat 2.7M THB revenue and 0.7M THB GP (25.9%) every month, no seasonality.** The note does not say which costs are inside GP -> compare GP-A/B/C before the P&L.

Per cup: packaging 6 THB (berry 8), labor 5-7 THB. Overhead 670k THB/month for 4 kitchens (140k-200k each).

? `base_price_thb` (45/49/49/65/89) is not in the data dictionary. Use it to check price / (1 - discount).

### 1.8 Do the order rows add up?

In [12]:
o = orders
n_total = len(o)
total_rev = o["gross_revenue_thb"].sum()

# 0.01 THB tolerance for float rounding.
rev_diff = o["gross_revenue_thb"] - o["units_sold"] * o["price_thb"]
bad_revenue = ~np.isclose(rev_diff, 0, atol=0.01)

# File and pandas both use 0 = Monday.
bad_dow = o["day_of_week"] != o["date"].dt.dayofweek

checks = {
    "revenue != units x price": bad_revenue,
    "units_sold < 0": o["units_sold"] < 0,
    "units_sold = 0": o["units_sold"] == 0,
    "price_thb <= 0": o["price_thb"] <= 0,
    "day_of_week does not match date": bad_dow,
    # keep="first" marks only the extra copies, so real rows are not counted twice.
    "exact duplicate rows (extra copies)": o.duplicated(keep="first"),
}

issue_table = pd.DataFrame([
    {"check": name,
     "rows": int(mask.sum()),
     "pct_rows": round(mask.mean() * 100, 3),
     "revenue_thb": round(o.loc[mask, "gross_revenue_thb"].sum(), 2),
     "pct_revenue": round(o.loc[mask, "gross_revenue_thb"].sum() / total_rev * 100, 3)}
    for name, mask in checks.items()
]).set_index("check")

print(f"orders_hourly: {n_total:,} rows | raw revenue {total_rev:,.2f} THB\n")
display(issue_table)

if bad_revenue.any():
    print(f"\nSum of (revenue - units x price) on mismatch rows: {rev_diff[bad_revenue].sum():,.2f} THB")
    display(o.loc[bad_revenue].head(10))

dup_mask = checks["exact duplicate rows (extra copies)"]
if dup_mask.any():
    by_name = o.loc[dup_mask, "item_name_raw"].value_counts()
    print(f"\nExact duplicate copies spread over {len(by_name)} names. Top 10:")
    print(by_name.head(10).to_string())

orders_hourly: 124,397 rows | raw revenue 31,276,898.60 THB



,rows,pct_rows,revenue_thb,pct_revenue
check,,,,
revenue != units x price,0,0.000,0.00,0.000
units_sold < 0,0,0.000,0.00,0.000
units_sold = 0,0,0.000,0.00,0.000
price_thb <= 0,85,0.068,0.00,0.000
day_of_week does not match date,0,0.000,0.00,0.000
exact duplicate rows (extra copies),372,0.299,91488.03,0.293



Exact duplicate copies spread over 30 names. Top 10:
item_name_raw
Guava Smoothie        21
ฝรั่งปั่น             21
แตงโมปั่น (ไซส์ M)    20
pineapple ice         19
guava ice             19
เสาวรสปั่น            18
สัปปะรดปั่น           18
Passion Fruit Ice     16
Pineapple Smoothie    15
Guava Ice             15


In [13]:
zero = o[o["price_thb"] <= 0]
display(zero.groupby(["item_name_raw", "rate_code"])
        .agg(rows=("units_sold", "size"), units=("units_sold", "sum")))
print(f"Price-0 rows: {len(zero)} | cups: {zero['units_sold'].sum()} "
      f"| all non-menu names: {zero['item_name_raw'].isin(NON_MENU).all()}")

# How often does each duplicated row appear? 2 = one real row + one copy.
dup_groups = o[o.duplicated(keep=False)].groupby(list(o.columns)).size()
print("\nTimes a duplicated row appears -> number of groups:")
print(dup_groups.value_counts().to_string())

# An hourly summary should have one row per key. Same key with different numbers would mean real extra rows.
key = ["date", "hour", "kitchen", "item_name_raw", "rate_code"]
key_only = o.duplicated(key, keep=False) & ~o.duplicated(keep=False)
print(f"\nRows sharing a key but not exact copies: {key_only.sum()}")

copies = o[dup_mask]
display(pd.DataFrame({
    "copies": [copies["rate_code"].str[:2].eq("LM").mean() * 100, copies["units_sold"].mean()],
    "all rows": [o["rate_code"].str[:2].eq("LM").mean() * 100, o["units_sold"].mean()],
}, index=["LINE MAN % of rows", "cups per row"]).round(1))
print(f"Copies: {len(copies)} rows, {copies['units_sold'].sum():,} cups, "
      f"{copies['gross_revenue_thb'].sum():,.0f} THB")

assert zero["item_name_raw"].isin(NON_MENU).all(), "price-0 row outside the non-menu names"
assert (dup_groups == 2).all(), "a row appears more than twice"
assert key_only.sum() == 0, "same key with different numbers"

rows  units
item_name_raw          rate_code             
Combo Set A (2 แก้ว)   GB-RC000     14     21
                       LM-RC000     19     35
น้ำส้มปั่น (เมนูทดลอง) GB-RC000     12     26
                       LM-RC000     13     29
แก้วเปล่า              GB-RC000     14     31
                       LM-RC000     13     32

Price-0 rows: 85 | cups: 174 | all non-menu names: True



Times a duplicated row appears -> number of groups:
2    372



Rows sharing a key but not exact copies: 0


,copies,all rows
LINE MAN % of rows,55.4,55.6
cups per row,4.9,5.0


Copies: 372 rows, 1,817 cups, 91,488 THB


**Raw: 124,397 rows, 31.28M THB.** Revenue = units x price on every row, no zero/negative units, weekday always matches date. OK.

**85 rows at price 0 (0.07%, 174 cups, 0 THB).** All are the 3 non-menu names at RC000 -> gone when non-menu items are dropped.

**372 exact duplicate copies (0.30% of rows, 1.8k cups, 91.5k THB).** No other row shares a date-hour-kitchen-name-code key, and copies match the overall platform and cup mix -> double export. Decision: drop copies, keep the first row.

## 2. What is left after cleaning?

### 2.1 What does each cleaning step remove?

In [14]:
import sys

sys.path.insert(0, str(ROOT))  # lets the notebook import src/
from src.load_clean import SKUS, clean_orders, load_mapping, load_tables, reconcile, week_monday

mapping = load_mapping()
print(mapping["sku"].value_counts().to_string())

clean, orders_log = clean_orders(sheets, mapping)
tables, tables_log = load_tables(sheets, return_log=True)
dq_log = pd.concat([orders_log, tables_log], ignore_index=True)
with pd.option_context("display.max_colwidth", None):
    display(dq_log)

sku
Watermelon           7
Pineapple            6
Guava                5
MixedBerryPremium    5
PassionFruit         5
DROP                 3


,table,category,issue,rows,pct_rows,revenue_thb,revenue_change_thb,cost_thb,effect,action
0,orders_hourly,duplicate,Exact duplicate rows (extra copies),372,0.299,91488.03,0.0,0.00,removed,"Drop, keep first row"
1,orders_hourly,naming,"31 raw names (language, typos, case, spaces)",124025,99.701,31185410.57,0.0,0.00,fixed,Map with item_mapping.csv to 5 SKUs or DROP
2,orders_hourly,not a sale,"Non-menu names (test menu, empty cup, combo)",370,0.297,38478.00,0.0,0.00,removed,Drop
3,orders_hourly,invalid value,units_sold <= 0 or price_thb <= 0,0,0.000,0.00,0.0,0.00,removed,Drop
4,orders_hourly,invalid value,gross_revenue_thb != units x price,0,0.000,0.00,0.0,0.00,fixed,Recalculate as units x price
5,orders_hourly,code,rate_code not in the dim tables,0,0.000,0.00,0.0,0.00,flagged,"Keep, flag_unknown_code"
6,orders_hourly,code,RC101 SKU part does not match the item,0,0.000,0.00,0.0,0.00,flagged,"Keep, flag_rc101_mismatch"
7,orders_hourly,not a sale,Berry rows before launch 2026-03-01,0,0.000,0.00,0.0,0.00,removed,Drop
8,orders_hourly,same key,Rows sharing date-hour-kitchen-sku-rate_code,0,0.000,0.00,0.0,0.00,info,"Keep, sum by key in the enrich step"
9,weekly_fruit_cost,missing,No berry fruit cost for launch week 2026-02-23,1,0.417,0.00,0.0,0.00,filled,Copy week 2026-03-02 (33.69 THB per cup)


**Orders: 372 duplicate copies + 370 non-menu rows removed = 742 rows, 130k THB (0.42%).** 370, not 374: 4 non-menu rows were also duplicate copies, so step 1 already removed them.

Zero rows for bad values, unknown codes, RC101 mismatch, berry before launch and same key. Same key = 0 means one spelling per SKU per hour, so the enrich step has nothing to sum.

**Tables: 1 fruit cost row, 4 waste costs and 6 waste rows filled (+4.5k THB waste),** all with `is_filled=True`.

### 2.2 Does the clean data reconcile with the raw data?

In [15]:
rec = reconcile(sheets["orders_hourly"], clean, dq_log)
display(rec)

removed = orders_log.set_index("issue")["rows"]
assert removed["Exact duplicate rows (extra copies)"] == 372
assert removed["Non-menu names (test menu, empty cup, combo)"] == 370
assert (clean["price_thb"] > 0).all() and (clean["units_sold"] > 0).all(), "zero price or units left"
assert len(clean) == 123_655
assert abs(rec.loc["Difference", "revenue_thb"]) <= 1, "raw - removed != clean revenue"
assert sorted(clean["sku"].unique()) == sorted(SKUS)
print("Orders asserts passed.")

,rows,revenue_thb
line,,
Raw orders_hourly,124397,31276898.60
- Exact duplicate rows (extra copies),-372,-91488.03
"- Non-menu names (test menu, empty cup, combo)",-370,-38478.00
- units_sold <= 0 or price_thb <= 0,0,0.00
- Berry rows before launch 2026-03-01,0,0.00
+ Revenue recalculated as units x price,0,0.00
Expected clean,123655,31146932.57
Actual clean,123655,31146932.57
Difference,0,0.00


Orders asserts passed.


**Raw 31.28M THB - 130k THB removed = 31.15M THB clean, difference 0.00 THB.** Rows: 124,397 - 742 = 123,655. 5 SKUs only. OK.

### 2.3 Do the clean tables join and add up?

In [16]:
raw_waste = sheets["waste_daily"]["est_waste_cost_thb"].sum()
filled_waste = tables_log.loc[tables_log["table"] == "waste_daily", "cost_thb"].sum()
clean_waste = tables["waste_daily"]["est_waste_cost_thb"]
print(f"Waste cost: raw {raw_waste:,.0f} + filled {filled_waste:,.0f} = {clean_waste.sum():,.0f} THB")
assert np.isclose(clean_waste.sum(), raw_waste + filled_waste)
assert clean_waste.notna().all()

# Every clean order must find its fruit cost, commission, rate code and cup cost.
orders_wk = clean.assign(week_start=week_monday(clean["date"]))
joins = {
    "weekly_fruit_cost": (orders_wk, tables["weekly_fruit_cost"][["week_start", "sku"]]),
    "platform_commission_rate": (clean, tables["platform_commission_rate"][["platform_code"]]),
    "rate_code_dim": (clean, tables["rate_code_dim"][["base_code"]]),
    "unit_cost": (clean, tables["unit_cost"][["sku"]]),
}
for name, (left, right) in joins.items():
    j = left.merge(right, how="left", indicator=True, validate="many_to_one")
    n_miss = int((j["_merge"] == "left_only").sum())
    print(f"{name}: {n_miss} orders without a match")
    assert n_miss == 0

Waste cost: raw 855,848 + filled 4,459 = 860,307 THB
weekly_fruit_cost: 0 orders without a match
platform_commission_rate: 0 orders without a match
rate_code_dim: 0 orders without a match
unit_cost: 0 orders without a match


**Waste: 855.8k raw + 4.5k filled = 860.3k THB, no blanks.** Every clean order joins fruit cost (launch week 2026-02-23 included), commission, rate code and cup cost. OK.

### 2.4 What is saved to `data/processed/`?

In [17]:
out = ROOT / "data" / "processed"
clean.to_parquet(out / "orders_clean.parquet", index=False)
dq_log.to_csv(out / "data_quality_log.csv", index=False, encoding="utf-8")

saved = pd.read_parquet(out / "orders_clean.parquet")
assert saved.shape == clean.shape
assert np.isclose(saved["gross_revenue_thb"].sum(), clean["gross_revenue_thb"].sum())
print(f"orders_clean.parquet: {saved.shape[0]:,} rows x {saved.shape[1]} columns")
print(f"data_quality_log.csv: {len(dq_log)} rows")

# The app reads outputs/ only, so the cleaning tables go there too.
app_out = ROOT / "outputs"
dq_log.to_csv(app_out / "dq_log.csv", index=False, encoding="utf-8")
rec.rename_axis("line").reset_index().to_csv(app_out / "reconcile.csv", index=False, encoding="utf-8")
raw_names = sheets["orders_hourly"].assign(item_name_raw=lambda d: d["item_name_raw"].str.strip())
name_map = (raw_names.groupby("item_name_raw")
            .agg(rows=("units_sold", "size"), cups=("units_sold", "sum"), revenue_thb=("gross_revenue_thb", "sum"))
            .join(mapping.set_index("item_name_raw")["sku"]).reset_index().sort_values(["sku", "rows"]))
assert name_map["sku"].notna().all() and len(name_map) == 31
name_map.to_csv(app_out / "item_mapping.csv", index=False, encoding="utf-8")
print(f"outputs/: dq_log {len(dq_log)} rows, reconcile {len(rec)} lines, item_mapping {len(name_map)} names")

orders_clean.parquet: 123,655 rows x 15 columns


data_quality_log.csv: 12 rows
outputs/: dq_log 12 rows, reconcile 9 lines, item_mapping 31 names


Clean reference tables are not saved: `load_tables()` rebuilds them from the raw file each time.

## Findings -> next steps
1. 742 rows removed (372 copies + 370 non-menu, 130k THB, 0.42%) -> 123,655 clean rows, reconcile difference 0.
2. 31 names -> 5 SKUs. 0 unknown codes, 0 RC101 mismatches, 0 same-key rows -> no summing needed in enrich.
3. Waste 860.3k THB = 855.8k raw + 4.0k launch day + 0.5k missing days (all `is_filled`).
4. Every clean order joins fruit cost, commission, rate code and cup cost -> ready for `src/economics.py` (CM per cup).
5. Budget GP 25.9% has no cost definition -> compare GP-A/B/C before building the P&L.